# TCGA-BRCA Baseline Analysis Prep V1 Review

This notebook is review-only. It loads the latest saved baseline-analysis-prep v1 outputs from disk, validates the saved run log, summarizes retained fields and missingness, and writes review tables for human audit.

## Load the latest saved baseline-analysis-prep run

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


def parse_json_list(value: str) -> list[str]:
    if value == '':
        return []
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise ValueError(f'Expected a JSON list, received: {value}')
    return [str(item) for item in parsed]


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'analysis-prep'
    / 'tcga_brca_baseline_analysis_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest baseline analysis prep pointer not found: {latest_pointer_path}. Run the prep script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
baseline_path = repo_root / latest_pointer['baseline_analysis_v1_tsv']
spec_path = repo_root / latest_pointer['baseline_analysis_v1_spec_tsv']
missingness_path = repo_root / latest_pointer['baseline_analysis_v1_missingness_tsv']
summary_path = repo_root / latest_pointer['baseline_analysis_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_pointer]))


## Load saved baseline-analysis-prep artifacts

In [ ]:
baseline_df = read_tsv(baseline_path)
spec_df = read_tsv(spec_path)
missingness_df = read_tsv(missingness_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if run_log.get('status') != 'completed':
    raise ValueError(f"Baseline analysis prep run is not completed: status={run_log.get('status')}")
if not run_log.get('validation', {}).get('passed', False):
    raise ValueError('Baseline analysis prep validation did not pass. Review the saved run_log.json before continuing.')

print(f"Baseline analysis prep run ID: {latest_pointer['baseline_analysis_v1_run_id']}")
print(f"Cohort v1 build ID: {latest_pointer['cohort_v1_build_id']}")
print(f"Processed baseline TSV: {baseline_path}")
print(f"Spec TSV: {spec_path}")
print(f"Missingness TSV: {missingness_path}")
print(f"Summary TSV: {summary_path}")
print(f"Run log: {run_log_path}")

validation_df = pd.DataFrame([run_log.get('validation', {})])
display(validation_df)


## Review retained fields, missingness, coverage, and carried ambiguity flags

In [ ]:
baseline_sorted_df = (
    baseline_df.assign(
        baseline_analysis_v1_row_id_numeric=pd.to_numeric(
            baseline_df['baseline_analysis_v1_row_id'], errors='raise'
        )
    )
    .sort_values('baseline_analysis_v1_row_id_numeric')
    .drop(columns='baseline_analysis_v1_row_id_numeric')
    .reset_index(drop=True)
)
spec_review_df = spec_df.sort_values(
    ['retained_in_baseline_analysis_v1', 'field_category', 'source_origin', 'field_name'],
    ascending=[False, True, True, True],
).reset_index(drop=True)
missingness_review_df = (
    missingness_df.assign(
        missing_like_fraction_numeric=pd.to_numeric(
            missingness_df['missing_like_fraction'], errors='raise'
        )
    )
    .sort_values(['missing_like_fraction_numeric', 'field_name'], ascending=[False, True])
    .drop(columns='missing_like_fraction_numeric')
    .reset_index(drop=True)
)
summary_review_df = summary_df.sort_values(['summary_section', 'summary_metric']).reset_index(drop=True)

retained_field_category_summary_df = (
    spec_review_df.loc[spec_review_df['retained_in_baseline_analysis_v1'] == 'yes']
    .groupby('field_category', dropna=False)
    .size()
    .reset_index(name='retained_field_count')
    .sort_values(['retained_field_count', 'field_category'], ascending=[False, True])
    .reset_index(drop=True)
)

ambiguity_flags_review_df = (
    baseline_sorted_df[
        [
            'baseline_analysis_v1_row_id',
            'provisional_patient_row_id',
            'bcr_patient_barcode',
            'bcr_patient_uuid',
            'ambiguity_note_flags_json',
        ]
    ]
    .assign(ambiguity_note_flag=lambda df: df['ambiguity_note_flags_json'].map(parse_json_list))
    .explode('ambiguity_note_flag')
    .drop(columns='ambiguity_note_flags_json')
    .reset_index(drop=True)
)

join_flag_lists = baseline_sorted_df['join_status_flags_json'].map(parse_json_list)
followup_match_counts = pd.to_numeric(baseline_sorted_df['followup_match_row_count'], errors='raise')
biospecimen_match_counts = pd.to_numeric(
    baseline_sorted_df['biospecimen_sample_match_row_count'], errors='raise'
)

followup_coverage_rows = [
    {
        'coverage_section': 'followup',
        'coverage_metric': 'final_row_count',
        'coverage_value': str(len(baseline_sorted_df)),
        'notes': 'Total patient-level rows reviewed in baseline analysis prep v1.',
    },
    {
        'coverage_section': 'followup',
        'coverage_metric': 'rows_with_followup_match',
        'coverage_value': str(int((baseline_sorted_df['row_has_followup_match'] == 'yes').sum())),
        'notes': 'Rows where followup_match_row_count > 0.',
    },
    {
        'coverage_section': 'followup',
        'coverage_metric': 'rows_without_followup_match',
        'coverage_value': str(int((baseline_sorted_df['row_has_followup_match'] == 'no').sum())),
        'notes': 'Rows where followup_match_row_count == 0.',
    },
    {
        'coverage_section': 'followup',
        'coverage_metric': 'rows_with_followup_multirow_signal',
        'coverage_value': str(int(sum('followup_multirow' in flags for flags in join_flag_lists))),
        'notes': 'Rows where join_status_flags_json includes followup_multirow.',
    },
    {
        'coverage_section': 'followup',
        'coverage_metric': 'followup_match_row_count_total',
        'coverage_value': str(int(followup_match_counts.sum())),
        'notes': 'Total grouped follow-up source rows represented across patient rows.',
    },
    {
        'coverage_section': 'followup',
        'coverage_metric': 'max_followup_match_row_count',
        'coverage_value': str(int(followup_match_counts.max())),
        'notes': 'Largest grouped follow-up row count observed for a single patient row.',
    },
]
for match_count, row_count in followup_match_counts.value_counts().sort_index().items():
    followup_coverage_rows.append(
        {
            'coverage_section': 'followup_match_distribution',
            'coverage_metric': f'followup_match_row_count_{int(match_count)}',
            'coverage_value': str(int(row_count)),
            'notes': 'Number of patient rows with the given follow-up grouped match count.',
        }
    )
followup_coverage_df = pd.DataFrame(followup_coverage_rows)

biospecimen_coverage_rows = [
    {
        'coverage_section': 'biospecimen',
        'coverage_metric': 'final_row_count',
        'coverage_value': str(len(baseline_sorted_df)),
        'notes': 'Total patient-level rows reviewed in baseline analysis prep v1.',
    },
    {
        'coverage_section': 'biospecimen',
        'coverage_metric': 'rows_with_biospecimen_sample_match',
        'coverage_value': str(int((baseline_sorted_df['row_has_biospecimen_sample_match'] == 'yes').sum())),
        'notes': 'Rows where biospecimen_sample_match_row_count > 0.',
    },
    {
        'coverage_section': 'biospecimen',
        'coverage_metric': 'rows_without_biospecimen_sample_match',
        'coverage_value': str(int((baseline_sorted_df['row_has_biospecimen_sample_match'] == 'no').sum())),
        'notes': 'Rows where biospecimen_sample_match_row_count == 0.',
    },
    {
        'coverage_section': 'biospecimen',
        'coverage_metric': 'rows_with_biospecimen_sample_multirow_signal',
        'coverage_value': str(int(sum('biospecimen_sample_multirow' in flags for flags in join_flag_lists))),
        'notes': 'Rows where join_status_flags_json includes biospecimen_sample_multirow.',
    },
    {
        'coverage_section': 'biospecimen',
        'coverage_metric': 'biospecimen_sample_match_row_count_total',
        'coverage_value': str(int(biospecimen_match_counts.sum())),
        'notes': 'Total grouped biospecimen_sample source rows represented across patient rows.',
    },
    {
        'coverage_section': 'biospecimen',
        'coverage_metric': 'max_biospecimen_sample_match_row_count',
        'coverage_value': str(int(biospecimen_match_counts.max())),
        'notes': 'Largest grouped biospecimen sample row count observed for a single patient row.',
    },
]
for match_count, row_count in biospecimen_match_counts.value_counts().sort_index().items():
    biospecimen_coverage_rows.append(
        {
            'coverage_section': 'biospecimen_match_distribution',
            'coverage_metric': f'biospecimen_sample_match_row_count_{int(match_count)}',
            'coverage_value': str(int(row_count)),
            'notes': 'Number of patient rows with the given biospecimen sample grouped match count.',
        }
    )
biospecimen_coverage_df = pd.DataFrame(biospecimen_coverage_rows)

display(summary_review_df)
display(retained_field_category_summary_df)
display(missingness_review_df.head(25))
display(followup_coverage_df)
display(biospecimen_coverage_df)


## Save review tables

In [ ]:
baseline_preview_df = baseline_sorted_df.head(100).reset_index(drop=True)

baseline_preview_path = results_root / '73_baseline_analysis_v1_preview.tsv'
spec_review_path = results_root / '74_baseline_analysis_v1_spec.tsv'
missingness_review_path = results_root / '75_baseline_analysis_v1_missingness.tsv'
followup_coverage_path = results_root / '76_baseline_analysis_v1_followup_coverage.tsv'
biospecimen_coverage_path = results_root / '77_baseline_analysis_v1_biospecimen_coverage.tsv'
ambiguity_flags_review_path = results_root / '78_baseline_analysis_v1_ambiguity_flags.tsv'
summary_review_path = results_root / '79_baseline_analysis_v1_summary.tsv'

baseline_preview_df.to_csv(baseline_preview_path, sep='\t', index=False)
spec_review_df.to_csv(spec_review_path, sep='\t', index=False)
missingness_review_df.to_csv(missingness_review_path, sep='\t', index=False)
followup_coverage_df.to_csv(followup_coverage_path, sep='\t', index=False)
biospecimen_coverage_df.to_csv(biospecimen_coverage_path, sep='\t', index=False)
ambiguity_flags_review_df.to_csv(ambiguity_flags_review_path, sep='\t', index=False)
summary_review_df.to_csv(summary_review_path, sep='\t', index=False)

print(f'Saved: {baseline_preview_path}')
print(f'Saved: {spec_review_path}')
print(f'Saved: {missingness_review_path}')
print(f'Saved: {followup_coverage_path}')
print(f'Saved: {biospecimen_coverage_path}')
print(f'Saved: {ambiguity_flags_review_path}')
print(f'Saved: {summary_review_path}')

display(baseline_preview_df)
display(summary_review_df)


This notebook remains review-only. It does not parse raw files, rebuild the cohort from source supplements, freeze a final endpoint, add treatment detail back in, expand child biospecimen layers, or perform modeling.